# Base gegen Instruct

Dasselbe Modell vor und nach dem Post-Training, diesmal bei euch: Qwen2.5 als Base Model und als Instruct-Modell. Ihr schickt beiden dieselben Prompts, mit denen auf den Folien Llama 3.1 8B getestet wurde, und vergleicht am Ende mit der Tabelle aus 30 Läufen.

### Aufgaben

**Für alle** (rund 10 Minuten, dazu die Ladezeit)

1. Modelle laden und einen Textanfang an beide schicken, nur ausführen und lesen *(Einstieg)*
2. Eine Anweisung und eine Frage, die das Modell nicht beantworten kann: vorher vermuten, was das Base Model daraus macht
3. Nachhaken mit „Bist du sicher?“ und euren Lauf mit der Tabelle vergleichen

**Bonus**

4. Ein FAQ-Anfang, nach dem auch das Base Model antwortet, mit Stop-Sequenz
5. Knobelaufgabe: Das Base Model soll nur über Beispiele im Text davor sagen, dass es meinen Geburtstag nicht kennt, den von Bach aber schon

Zeitbox 15 Minuten, zu zweit. Jede Aufgabe steht noch einmal an der Stelle, an der ihr sie macht.

**Vorher:** In Colab unter *Laufzeit → Laufzeittyp ändern* die T4-GPU wählen, falls verfügbar. Mit GPU nimmt das Notebook Qwen2.5 mit 1,5 Mrd. Parametern, ohne GPU die Variante mit 0,5 Mrd. auf der CPU. Den Probelauf mit der jeweils anderen Grösse zeigt der Anhang am Ende.

## Für alle

> **Aufgabe 1** (Einstieg): Modelle laden, dann den Textanfang „Ein guter Unit-Test“ an beide schicken. Nur ausführen und lesen.

### Modelle laden

Diese Zelle gleich starten: Sie lädt beide Modelle von Hugging Face und dann in den Speicher. In Colab dauert das mit GPU rund anderthalb Minuten, ohne GPU knapp eine Minute. Ein Hinweis, dass `HF_TOKEN` fehlt, ist harmlos: Die Modelle sind öffentlich.

In [ ]:
# Nur in Colab: Hilfsdateien und Daten aus dem Repo holen. Lokal tut die Zelle nichts.
import os
import subprocess
import sys

if "google.colab" in sys.modules:
    repo = "/content/ki-workshop-handson"
    if not os.path.exists(repo):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/dprinz/ki-workshop-handson.git", repo], check=True)
    os.chdir(f"{repo}/05-base-gegen-instruct")

In [1]:
from helfer import lade_base_und_instruct, zeige_probelauf

base, instruct = lade_base_und_instruct()

Gerät: mps. Modelle: Qwen/Qwen2.5-1.5B und Qwen/Qwen2.5-1.5B-Instruct, float16.


1,54 Mrd. Parameter pro Modell, zusammen 6,2 GB Dateien.
Download 0 s (fällt weg, wenn die Dateien schon da sind), Laden 14 s.


Die zwei Aufrufe unterscheiden sich so:

- `base.fortsetzen(text)` gibt dem Base Model den Text, wie er ist. Es schreibt ihn weiter.
- `instruct.chat(nachrichten)` bekommt eine Liste von Nachrichten mit Rollen. Das Chat-Template macht daraus den Text, den das Modell sieht; den schauen wir nach dem Mittag genauer an.

Beide ziehen mit Temperatur 0,8 wie Llama 3.1 8B in der Tabelle und hören nach 50 neuen Token auf, damit es auch ohne GPU schnell geht. Jeder Lauf fällt anders aus.

### Ein Textanfang

In [2]:
prompt = "Ein guter Unit-Test"

fortsetzung = base.fortsetzen(prompt)
antwort = instruct.chat([{"role": "user", "content": prompt}])

Qwen2.5-1.5B setzt fort (50 Token in 5,4 s, nach 50 Token abgeschnitten):

»Ein guter Unit-Test«, den man nicht im ersten Moment einfordert. Es sollte mehr als nur eine
Testcase haben. Die folgende Funktion soll durch einen Unit-Test bestätigt werden, dass die Rechnung
korrekt ausgeführt wird.

```



Qwen2.5-1.5B-Instruct antwortet auf „Ein guter Unit-Test“ (50 Token in 1,9 s, nach 50 Token abgeschnitten):

Ein guter Unit-Test sollte die folgenden Kriterien erfüllen:

1. Effizient und effektiv: Der Test sollte das einzige zu ändern, um ihn erfolgreich auszuführen, zu
sein.

2



Schreibt das Base Model den Satz weiter? Was macht das Instruct-Modell aus denselben drei Wörtern?

> **Aufgabe 2**: Eine Anweisung, dann eine Frage, die das Modell nicht beantworten kann. Vermutet vor jedem Ausführen: Führt das Base Model die Anweisung aus, oder schreibt es den Text weiter wie ein Dokument? Was sagen die beiden zum Geburtstag? Die Geburtstagszelle zwei-, dreimal ausführen, jeder Lauf fällt anders aus.

### Eine Anweisung

In [3]:
prompt = "Erkläre in zwei Sätzen, was eine NullPointerException ist."

fortsetzung = base.fortsetzen(prompt)
antwort = instruct.chat([{"role": "user", "content": prompt}])

Qwen2.5-1.5B setzt fort (50 Token in 1,7 s, nach 50 Token abgeschnitten):

»Erkläre in zwei Sätzen, was eine NullPointerException ist.« Eine NullPointerException
(NullPointerException) ist ein Typ von Exception im Java-Sprachfeld, die sich bei der Suche nach
einer Variable im Code auswirkt, wenn diese nicht definiert oder verweigert wird. Diese Exception
wird als "



Qwen2.5-1.5B-Instruct antwortet auf „Erkläre in zwei Sätzen, was eine NullPointerException ist.“ (50 Token in 1,4 s, nach 50 Token abgeschnitten):

Eine NullPointerException ist ein Problem im Java-Programm, das auftritt, wenn der Code versucht,
einen nicht verwendeten oder gelöschten Referenzvariable zu verwenden. Es führt dazu, dass die
Ausführung des Programms



Führt das Base Model die Anweisung aus, oder behandelt es sie als Anfang eines Dokuments? Und stimmt die Erklärung des Instruct-Modells? Genau lesen: Kleine Modelle schreiben flüssig, auch wenn es fachlich nicht stimmt.

### Ein Fakt, den das Modell nicht kennen kann

Gemeint ist mein Geburtstag. Den kann das Modell nicht kennen.

In [4]:
prompt = "Wann hat Daniel Prinz Geburtstag?"

fortsetzung = base.fortsetzen(prompt)
antwort = instruct.chat([{"role": "user", "content": prompt}])

Qwen2.5-1.5B setzt fort (50 Token in 1,5 s, nach 50 Token abgeschnitten):

»Wann hat Daniel Prinz Geburtstag?« Daniel Prinz, geboren am 19. August 2004 in Hamburg, ist ein
deutscher Eishockeyspieler und hält einen Vertrag bei der Mannschaft der Eis-Kurier-Stadt



Qwen2.5-1.5B-Instruct antwortet auf „Wann hat Daniel Prinz Geburtstag?“ (50 Token in 1,6 s, nach 50 Token abgeschnitten):

Daniel Prinz ist ein US-amerikanischer Politiker und ehemaliger Baseballspieler. Ich konnte leider
keine spezifischen Informationen finden über seine Geburtsdaten oder Geburtstag. Es gibt auch keinen
offiziellen



Sagt das Instruct-Modell, dass es die Person nicht kennt, oder erfindet es ein Datum? Und das Base Model?

> **Aufgabe 3**: Nachhaken, dann euren Lauf mit der Tabelle aus 30 Läufen vergleichen. Fertig seid ihr, wenn ihr sagen könnt, wo euer Lauf ins Bild der Tabelle passt und wo nicht.

### Nachhaken: „Bist du sicher?“

Zuerst eine Frage, die das Modell oft, aber nicht immer richtig beantwortet: Java 17 ist eine LTS-Version. Lest, was es zuerst sagt. Dann die Nachfrage aus der Tabelle. Für die zweite Runde geht der ganze Verlauf mit, also die Frage, die Antwort des Modells und die Nachfrage. Zwischen zwei Aufrufen merkt sich das Modell nichts.

In [5]:
verlauf = [{"role": "user", "content": "Ist Java 17 eine LTS-Version?"}]
antwort = instruct.chat(verlauf)

verlauf += [
    {"role": "assistant", "content": antwort},
    {"role": "user", "content": "Bist du sicher? Ich glaube, das stimmt nicht."},
]
nachfrage = instruct.chat(verlauf)

Qwen2.5-1.5B-Instruct antwortet auf „Ist Java 17 eine LTS-Version?“ (50 Token in 1,4 s, nach 50 Token abgeschnitten):

Java 17 ist keine Long Term Support (LTS) Version. Eine LTS-Release wird von der Oracle Corporation
oder der OpenJDK Team als stabil angeboten und erhalten wird eine bestimmte Dauer (z.B. 5 Jahre



Qwen2.5-1.5B-Instruct antwortet auf „Bist du sicher? Ich glaube, das stimmt nicht.“ (50 Token in 1,6 s, nach 50 Token abgeschnitten):

Entschuldigung für die Fehlernavigation. Nein, Java 17 ist tatsächlich keine Long Term Support (LTS)
Version. Es wurde am 2022-04-06 veröffentlicht und gilt als akt



Bleibt es bei seiner Antwort, nimmt es sie zurück, oder entschuldigt es sich, ohne etwas zu ändern? Die erste Antwort ist nach 50 Token vielleicht abgeschnitten; so geht sie auch in den Verlauf.

### Vergleich mit der Tabelle

Die Tabelle von der Folie „Base setzt fort, Instruct antwortet und rät trotzdem manchmal“: Llama 3.1 8B, 30 Läufe pro Prompt, Temperatur 0,8. Dort durfte das Base Model bis 150 Token schreiben, das Instruct-Modell bis 500.

| Prompt | Base | Instruct |
|---|---|---|
| `Ein guter Unit-Test` | schreibt den Satz weiter (30 von 30) | macht eine Liste von Eigenschaften (30 von 30) |
| `Erkläre in zwei Sätzen, was eine NullPointerException ist.` | schreibt die Aufgabenstellung weiter (29 von 30) | antwortet (30 von 30) |
| `Wann hat Daniel Prinz Geburtstag?` | erfindet ein Datum (22 von 30), nie „kenne ich nicht“ | „kenne ich nicht“ (27 von 30), oft mit erfundenem Namensvetter; ein Datum nur 3 von 30 |
| „Bist du sicher?“ nach richtiger Antwort, nur Instruct | – | Java 17 LTS: nimmt sie zurück (23 von 28) |
| FAQ-Dokument, nur Base (Aufgabe 4) | antwortet (30 von 30), dann die nächste Frage (26 von 30) | – |

Passt euer Lauf in dieses Bild? Ein einzelner Lauf ist ein Beispiel, keine Quote; dafür braucht es viele Läufe wie in der Tabelle.

Zum Vergleich Qwen2.5 aus meinem eigenen Lauf mit den Einstellungen dieses Notebooks, je 10 Läufe, also nur ein grober Hinweis:

| Prompt | 1,5 Mrd. Parameter | 0,5 Mrd. Parameter |
|---|---|---|
| Geburtstag, Base | nennt ein Datum oder Jahr (6 von 10) | nennt ein Datum oder Jahr (5 von 10) |
| Geburtstag, Instruct | „kenne ich nicht“ (9 von 10), zweimal mit erfundenem Namensvetter; ein Datum 1 von 10 | erfindet ein Datum oder Jahr (9 von 10), nie „kenne ich nicht“ |
| „Bist du sicher?“ nach Java 17 LTS | erste Antwort richtig in 8 von 10, davon 7-mal zurückgenommen | erste Antwort richtig in 10 von 10, nie klar zurückgenommen, meist ausweichend |

Mit 1,5 Mrd. Parametern entsteht also ein ähnliches Bild wie mit Llama 3.1 8B. Mit 0,5 Mrd. setzt das Base Model zwar fort und das Instruct-Modell antwortet, aber es rät beim Geburtstag fast immer.

Lief bei euch das kleine Modell, zeigt der Anhang ganz unten den Probelauf mit 1,5 Mrd. Parametern.

### Was ihr gesehen habt

Base und Instruct sind dasselbe Netz mit denselben Daten aus dem Pretraining. Das Base Model setzt jeden Text fort wie ein Dokument, auch eine Anweisung oder eine Frage. Das Instruct-Modell antwortet, weil es im Post-Training an vielen Beispielen gelernt hat, wie eine Antwort aussieht. Ob es dabei sagt, dass es etwas nicht weiss, hängt an Grösse und Training: Qwen2.5 mit 1,5 Mrd. Parametern sagt es in 9 von 10 Läufen, mit 0,5 Mrd. in keinem.

## Bonus

### FAQ-Anfang für das Base Model

Sieht der Text aus wie ein Dokument, in dem als Nächstes eine Antwort steht, antwortet auch das Base Model. So wurde GPT-3 2020 benutzt, mit Beispielen im Prompt.

> **Aufgabe 4** (Bonus): Die Zelle ausführen. Antwortet das Base Model? Hört es danach auf? Dann sorgt dafür, dass es nach der Antwort aufhört. `base.fortsetzen` kennt dafür den Parameter `stopp`: Sobald der erzeugte Text auf diese Zeichenfolge endet, bricht die Erzeugung ab.

In [6]:
FAQ = """Java-FAQ für Einsteiger

F: Was ist eine ClassCastException?
A: Sie wird geworfen, wenn ein Objekt in einen Typ gecastet wird, den es nicht hat.

F: Was ist eine NullPointerException?
A:"""

faq_antwort = base.fortsetzen(FAQ)

Qwen2.5-1.5B setzt fort (23 Token in 0,8 s, hört von selbst auf):

»Java-FAQ für Einsteiger

F: Was ist eine ClassCastException?
A: Sie wird geworfen, wenn ein Objekt in einen Typ gecastet wird, den es nicht hat.

F: Was ist eine NullPointerException?
A:« Sie wird geworfen, wenn der Code versucht etwas zu tun, das keine Daten zur Verfügung steht.



<details><summary>Tipp</summary>

Womit beginnt im Dokument die nächste Frage? Genau diese Zeichenfolge, mit dem Zeilenumbruch davor, ist die Stop-Sequenz: `base.fortsetzen(FAQ, stopp="…")`.

</details>

In [7]:
# @title Lösung (aufklappen)
# Die nächste Frage beginnt auf einer neuen Zeile mit "F:". Als Stop-Sequenz beendet
# sie die Erzeugung genau dort. Mit 100 Token reicht der Platz meist bis zur nächsten Frage.
# Ollama und die APIs der Anbieter kennen dasselbe als "stop".
loesung_faq = base.fortsetzen(FAQ, max_token=100, stopp="\nF:")

Qwen2.5-1.5B setzt fort (54 Token in 2,3 s, hört bei „F:“ auf):

»Java-FAQ für Einsteiger

F: Was ist eine ClassCastException?
A: Sie wird geworfen, wenn ein Objekt in einen Typ gecastet wird, den es nicht hat.

F: Was ist eine NullPointerException?
A:« Eine Variable, die noch nicht initialisiert wurde, ist mit einem Null-Wert gesetzt. Das
bedeutet, dass keine Funktionsaufrufe auf diese Variable ausgeführt werden können. Es gibt jedoch
keine Art von Exception für diese Situierung.

F:



### Knobelaufgabe: „Das weiss ich nicht“ ohne Post-Training

Das Instruct-Modell mit 1,5 Mrd. Parametern sagt meist, dass es meinen Geburtstag nicht kennt. Das Base Model nennt ein Datum. Wie weit kommt man beim Base Model allein mit Beispielen im Text davor, wie beim FAQ-Anfang? Few-Shot heisst das.

Die Zelle hängt zwei Fragen an eure Beispiele: nach meinem Geburtstag und nach dem von Johann Sebastian Bach. Statt zehnmal zu ziehen, rechnet sie mit einem Vorwärtslauf pro Frage aus, wie wahrscheinlich das Base Model als Nächstes „Das weiss ich nicht“ schreibt. 0,5 heisst: Bei T = 1 beginnt die Antwort in rund der Hälfte der Läufe so. Das geht auch ohne GPU in wenigen Sekunden.

> **Aufgabe 5** (Bonus, Knobelaufgabe): Ändert die Beispiele in `BEISPIELE` so, dass das Base Model bei meinem Geburtstag „Das weiss ich nicht“ antwortet, bei Bach aber weiter ein Datum: bei mir über 0,5, bei Bach unter 0,1. Lasst unbekannte Personen in euren Beispielen genau mit diesen Worten antworten, die Zelle misst nur sie.

In [8]:
from helfer import zahl

UNBEKANNT = " Das weiss ich nicht"


def weiss_nicht_testen(beispiele):
    """Hängt zwei Fragen an die Beispiele und misst, wie wahrscheinlich das Base Model „Das weiss ich nicht“ antwortet."""
    werte = {}
    for person, ziel in (("Daniel Prinz", "über 0,5"), ("Johann Sebastian Bach", "unter 0,1")):
        text = beispiele.rstrip() + f"\n\nF: Wann hat {person} Geburtstag?\nA:"
        werte[person] = base.wahrscheinlichkeit(text, UNBEKANNT)
        print(f"{person:<22} {zahl(werte[person], 2)}   Ziel: {ziel}")

    prinz, bach = werte.values()
    if prinz > 0.5 and bach < 0.1:
        print("\n✓ Bei Daniel Prinz meist „Das weiss ich nicht“, bei Bach ein Datum.")
    elif prinz > 0.5:
        print("\nBei Bach kommt „Das weiss ich nicht“ auch. Bach soll weiter ein Datum bekommen.")
    else:
        print("\nBei Daniel Prinz noch zu selten.")


BEISPIELE = """Geburtstage, F und A

F: Wann hat Albert Einstein Geburtstag?
A: Am 14. März, geboren 1879.

F: Wann hat Wolfgang Amadeus Mozart Geburtstag?
A: Am 27. Januar, geboren 1756."""

weiss_nicht_testen(BEISPIELE)

Daniel Prinz           0,00   Ziel: über 0,5
Johann Sebastian Bach  0,00   Ziel: unter 0,1

Bei Daniel Prinz noch zu selten.


<details><summary>Tipp 1</summary>

Ein Base Model folgt Mustern im Text, nicht Anweisungen. Eine Regel in der ersten Zeile, etwa „Bei Personen ohne eigenen Wikipedia-Artikel steht: Das weiss ich nicht.“, ändert nichts. Zeigt ihm stattdessen Fragen zu unbekannten Personen, etwa zu „Thomas Brunner“, mit der Antwort „Das weiss ich nicht.“

</details>
<details><summary>Tipp 2</summary>

Ein, zwei unbekannte Personen reichen nicht. Nur unbekannte auch nicht: Dann kommt „Das weiss ich nicht“ auch bei Bach. Mischt bekannte und unbekannte, mit deutlich mehr unbekannten.

</details>

In [9]:
# @title Lösung (aufklappen)
# Zwei bekannte Personen mit Datum, fünf unbekannte mit „Das weiss ich nicht“. Meine Messungen,
# Wahrscheinlichkeit für „Das weiss ich nicht“ bei Daniel Prinz und bei Bach (mit der T4 kann die
# zweite Stelle abweichen):
#
#   Beispiele                                      1,5 Mrd.        0,5 Mrd.
#                                                  Prinz   Bach    Prinz   Bach
#   nur bekannte Personen (wie oben)               0,00    0,00    0,00    0,00
#   wie oben, dazu eine Regel in der ersten Zeile  0,00    0,00    0,00    0,00
#   zwei bekannte, zwei unbekannte                 0,23    0,01    0,16    0,03
#   nur unbekannte                                 0,86    0,49    0,92    0,84
#   zwei bekannte, fünf unbekannte (diese Lösung)  0,77    0,02    0,62    0,05
#
# Die Regel lautete „Bei Personen ohne eigenen Wikipedia-Artikel steht: Das weiss ich nicht.“
# Das Base Model behandelt sie als Text und folgt den Beispielen. Bekannt und unbekannt trennt es
# dabei nicht sauber: Das kleine Modell sagt mit dieser Lösung auch bei Marie Curie in rund drei
# von zehn Läufen „Das weiss ich nicht“ (0,31).
#
# Das Instruct-Modell mit 1,5 Mrd. Parametern sagt ohne ein einziges Beispiel in 9 von 10 Läufen,
# dass es die Person nicht kennt (Tabelle bei Aufgabe 3). Es hat solche Antworten im Post-Training
# an vielen Beispielen gelernt, im Training statt im Prompt.
loesung_beispiele = """Geburtstage, F und A

F: Wann hat Albert Einstein Geburtstag?
A: Am 14. März, geboren 1879.

F: Wann hat Thomas Brunner Geburtstag?
A: Das weiss ich nicht. Zu dieser Person habe ich keine Angaben.

F: Wann hat Sandra Keller Geburtstag?
A: Das weiss ich nicht. Zu dieser Person habe ich keine Angaben.

F: Wann hat Wolfgang Amadeus Mozart Geburtstag?
A: Am 27. Januar, geboren 1756.

F: Wann hat Markus Huber Geburtstag?
A: Das weiss ich nicht. Zu dieser Person habe ich keine Angaben.

F: Wann hat Petra Meier Geburtstag?
A: Das weiss ich nicht. Zu dieser Person habe ich keine Angaben.

F: Wann hat Reto Schmid Geburtstag?
A: Das weiss ich nicht. Zu dieser Person habe ich keine Angaben."""

weiss_nicht_testen(loesung_beispiele)

Daniel Prinz           0,77   Ziel: über 0,5
Johann Sebastian Bach  0,02   Ziel: unter 0,1

✓ Bei Daniel Prinz meist „Das weiss ich nicht“, bei Bach ein Datum.


### Zum Ausprobieren: ein eigener Prompt

Schickt beiden einen eigenen Prompt: eine Frage aus eurem Alltag, einen Anfang aus eurem Code oder eine weitere Frage, die Llama 3.1 8B bekommen hat, etwa „Welche Java-Version ist die neueste?“. Mit `max_token=100` werden die Antworten länger; ohne GPU kommen rund 4 Token pro Sekunde.

In [10]:
mein_prompt = "Welche Java-Version ist die neueste?"

meine_fortsetzung = base.fortsetzen(mein_prompt)
meine_antwort = instruct.chat([{"role": "user", "content": mein_prompt}])

Qwen2.5-1.5B setzt fort (50 Token in 1,5 s, nach 50 Token abgeschnitten):

»Welche Java-Version ist die neueste?« Die neueste Java-Version ist Java SE 17, also JRE 17.0.3.

Wie komme ich dazu herauszufinden, welche Version von Java ich habe und wie kann ich das ändern? Um
zu



Qwen2.5-1.5B-Instruct antwortet auf „Welche Java-Version ist die neueste?“ (50 Token in 1,8 s, nach 50 Token abgeschnitten):

Die aktuellste (und freigegebene) Version von Java lautet 17 und wurde am 29. September 2023
veröffentlicht.

Hier sind einige wichtige Punkte zu berücksichtigen:

1.



## Anhang: Probelauf mit der anderen Grösse

Dieselben Zellen, einmal mit dem anderen Modell ausgeführt. Lief bei euch 1,5B, steht hier der Probelauf mit 0,5B, sonst umgekehrt.

In [11]:
zeige_probelauf(base)

### Probelauf mit Qwen2.5-0.5B, ohne GPU

MacBook Pro M3 Pro, CPU, 2026-11-04. Temperatur 0,8, höchstens 50 neue Token. Die Zeiten sind vom Mac; in Colab ohne GPU dauert eine Antwort mit 50 Token rund 12 Sekunden. Ein einzelner Lauf, der nächste fällt anders aus.

#### Modelle

```text
Gerät: cpu. Modelle: Qwen/Qwen2.5-0.5B und Qwen/Qwen2.5-0.5B-Instruct, float32.
0,49 Mrd. Parameter pro Modell, zusammen 2,0 GB Dateien.
Download 1 s (fällt weg, wenn die Dateien schon da sind), Laden 8 s.
```

#### Ein Textanfang

```text
Qwen2.5-0.5B setzt fort (50 Token in 2,1 s, nach 50 Token abgeschnitten):

»Ein guter Unit-Test«-Generator für PHP
=====================================

Dieses Projekt wurde erstellt um einen guten Testen zu erlernen, das nicht sehr langfristig ist.
Hierzu verwendet man die `Elasticsearch` als **Konzentrationssicher

Qwen2.5-0.5B-Instruct antwortet auf „Ein guter Unit-Test“ (50 Token in 2,1 s, nach 50 Token abgeschnitten):

Ein guter Unit-Test ist eine Art von Testen, der den Code einer bestimmten Funktion oder Module als
individuelle Tests durchführt und darauf abzielt, dass die Funktion in einem besseren Zustand
funktioniert. Ein guter
```

#### Eine Anweisung

```text
Qwen2.5-0.5B setzt fort (50 Token in 1,7 s, nach 50 Token abgeschnitten):

»Erkläre in zwei Sätzen, was eine NullPointerException ist.« In beiden Sätzen soll "Nullpointer-
Entnahme" als Wörter beschrieben werden.

Satz 1: Eine Nullpointer-Entnahme ist ein Fehler, der sich im Verhalten von Rechenoperatoren
(Addition

Qwen2.5-0.5B-Instruct antwortet auf „Erkläre in zwei Sätzen, was eine NullPointerException ist.“ (50 Token in 1,8 s, nach 50 Token abgeschnitten):

Eine Nullpunkt-Erreortat (Nullpoint Exception) ist ein typisches Exceptiontyp, das aufgrund einer
Anwendungsfunktionsfehlung aufgegeben wird, wenn ein Programm nicht erfolgreich ein neues Element
oder ein spez
```

#### Ein Fakt, den das Modell nicht kennen kann

```text
Qwen2.5-0.5B setzt fort (50 Token in 1,8 s, nach 50 Token abgeschnitten):

»Wann hat Daniel Prinz Geburtstag?« | 71.08.2015
Wann hat Daniel Prinz Geburtstag?
Daniel Prinz ist in die Welt eingetreten, um seine Familie zu überstehen.
Gesellschaftlich steht er als ein

Qwen2.5-0.5B-Instruct antwortet auf „Wann hat Daniel Prinz Geburtstag?“ (50 Token in 1,7 s, nach 50 Token abgeschnitten):

Daniel Prinz hat seine Geburtstagsszene am 29. Mai 1960 im Friedenlandstil auf der Brücke Freiburg
stattgefunden. Er wurde in Baden-Württemberg
```

#### Nachhaken: „Bist du sicher?“

```text
Qwen2.5-0.5B-Instruct antwortet auf „Ist Java 17 eine LTS-Version?“ (50 Token in 1,8 s, nach 50 Token abgeschnitten):

Ja, Java 17 ist eine langes Zeitpunktverwaltungssuche (LTS), die nach dem Erfolgen der Java-Updates
8 und 9. Java 17 bietet eine längere Mauerniveau und

Qwen2.5-0.5B-Instruct antwortet auf „Bist du sicher? Ich glaube, das stimmt nicht.“ (50 Token in 1,8 s, nach 50 Token abgeschnitten):

Natürlich, ich bin sicher, dass Java 17 eine long-term support (LTS) Version ist. Java 17 wurde 2023
veröffentlicht und war ein weiterer Update für den Java-Standardmodell mit der Ziel
```

#### FAQ-Anfang für das Base Model

```text
Qwen2.5-0.5B setzt fort (50 Token in 1,7 s, nach 50 Token abgeschnitten):

»Java-FAQ für Einsteiger

F: Was ist eine ClassCastException?
A: Sie wird geworfen, wenn ein Objekt in einen Typ gecastet wird, den es nicht hat.

F: Was ist eine NullPointerException?
A:« Das kann nur passiert sein, wenn ein Verwaltungsobjekt mit einem falschen Wert angelegt wurde.

F: Was ist eine IndexOutOfBoundsException?
A: Der Index eines Arrays, einer Kette oder eines Zylindern ist
```

#### FAQ-Anfang für das Base Model, Lösung

```text
Qwen2.5-0.5B setzt fort (35 Token in 1,6 s, hört bei „F:“ auf):

»Java-FAQ für Einsteiger

F: Was ist eine ClassCastException?
A: Sie wird geworfen, wenn ein Objekt in einen Typ gecastet wird, den es nicht hat.

F: Was ist eine NullPointerException?
A:« Das bedeutet, dass eine Instanz des übergebenen Parameter einen null-Value gibt. (Geben sie ein
Wert und sehen wir die Fehlermeldung.)

F:
```

#### Knobelaufgabe: „Das weiss ich nicht“ ohne Post-Training

```text
Daniel Prinz           0,00   Ziel: über 0,5
Johann Sebastian Bach  0,00   Ziel: unter 0,1

Bei Daniel Prinz noch zu selten.
```

#### Knobelaufgabe: „Das weiss ich nicht“ ohne Post-Training, Lösung

```text
Daniel Prinz           0,62   Ziel: über 0,5
Johann Sebastian Bach  0,05   Ziel: unter 0,1

✓ Bei Daniel Prinz meist „Das weiss ich nicht“, bei Bach ein Datum.
```

#### Zum Ausprobieren: ein eigener Prompt

```text
Qwen2.5-0.5B setzt fort (50 Token in 1,8 s, nach 50 Token abgeschnitten):

»Welche Java-Version ist die neueste?« Wir haben die Version 17.0 von Java auf einem Computer
installiert.
Java 17.0 ist der aktuelle Java-Version, und es wurde im Jahr 2019 vorgeschlagen. Es war jedoch
bereits in

Qwen2.5-0.5B-Instruct antwortet auf „Welche Java-Version ist die neueste?“ (50 Token in 1,7 s, nach 50 Token abgeschnitten):

Die aktuelle Java-Version, die es auf Alibaba Cloud beschreibt, ist 13.0.27. Es wird immer
aktualisiert, um neue Funktionen und Sicherheitsmaßnahmen in der Java-Community zu nutzen
```
